<a href="https://colab.research.google.com/github/watanabe3tipapa/GoogleColaboratory/blob/main/mcp/colab_gui_desktop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Colab GUI Desktop + Browser Automation Demo

このノートブックは以下を1本で実現します。
1. **仮想デスクトップ**（Xvfb）を立ち上げ、noVNC でブラウザから遠隔操作可能にする
2. **PyAutoGUI** で Python からマウス・キーボード操作・スクリーンショット取得
3. **Selenium / Playwright** でヘッドレス Chrome 自動操作

**想定環境**: Google Colab (無料 T4 でも CPU でも動作)

## ① 依存パッケージのインストール

- `xvfb`: 仮想ディスプレイ
- `x11vnc`: VNC サーバー
- `novnc`: ブラウザベース VNC クライアント
- `fluxbox`: 軽量ウィンドウマネージャー
- `chromium-browser`: ブラウザ（Selenium/Playwright 用）
- `scrot`: スクリーンショット（PyAutoGUI 補助）
- Python ライブラリ: `pyautogui`, `selenium`, `playwright`, `pyngrok`

In [ ]:
# Linux 依存関係のインストール
!apt-get update -qq
!apt-get install -y -qq \
  xvfb \
  x11vnc \
  novnc \
  websockify \
  fluxbox \
  chromium-browser \
  scrot \
  xfonts-base \
  ffmpeg

# Python ライブラリ
!pip install -q pyautogui selenium playwright pyngrok pillow
!playwright install-deps chromium
!playwright install chromium

print('インストール完了')

## ② 仮想デスクトップ + VNC + noVNC の起動

- 画面サイズ 1280x720 の仮想ディスプレイ `:99` を作成
- VNC サーバーをポート 5900 で待ち受け
- noVNC（WebSocket 経由）をポート 6080 で待ち受け
- 軽量ウィンドウマネージャー Fluxbox を起動

In [ ]:
import subprocess
import time
import os

# 既存プロセスのクリンアップ（再実行時用）
!pkill -f Xvfb; pkill -f x11vnc; pkill -f websockify; pkill -f fluxbox
time.sleep(1)

# 仮想ディスプレイ :99 を作成
xvfb_proc = subprocess.Popen([
    "Xvfb", ":99", "-screen", "0", "1280x720x24",
    "-ac", "+extension", "GLX", "+render", "-noreset"
])
os.environ["DISPLAY"] = ":99"
time.sleep(2)

# ウィンドウマネージャー起動
fluxbox_proc = subprocess.Popen(["fluxbox"])
time.sleep(1)

# VNC サーバー起動（パスワードなし・共有アクセス可）
vnc_proc = subprocess.Popen([
    "x11vnc",
    "-display", ":99",
    "-nopw",
    "-forever",
    "-shared",
    "-rfbport", "5900",
    "-noxkb",
    "-noxrecord",
    "-noxfixes",
    "-noxdamage",
    "-xrandr", "resize",
])
time.sleep(2)

# noVNC 起動（WebSocket -> VNC ブリッジ）
# /usr/share/novnc が存在しない環境もあるため、pip 経由や代替パスにフォールバック
novnc_web_path = "/usr/share/novnc"
if not os.path.isdir(novnc_web_path):
    novnc_web_path = "/usr/share/novnc/utils"
if not os.path.isdir(novnc_web_path):
    # フォールバック: 直接ダウンロードは不可なので既存パスを探す
    import shutil
    novnc_web_path = shutil.which("websockify")
    novnc_web_path = os.path.dirname(novnc_web_path) if novnc_web_path else "."

websockify_proc = subprocess.Popen([
    "websockify",
    "--web", novnc_web_path if os.path.isdir(novnc_web_path) else "/usr/share/novnc",
    "--cert", "self.pem",
    "localhost:6080",
    "localhost:5900"
])
time.sleep(2)

print("仮想デスクトップ起動完了")
print("ローカル noVNC URL: http://localhost:6080/vnc.html")
print("VNC ポート: localhost:5900")

## ③ 外部公開（トンネル）の選択

以下のいずれかを選んで実行してください。
- **A. ngrok**（無料でランダム URL。固定 URL が必要なら Hobby プラン）
- **B. Cloudflare Tunnel**（無料・固定サブドメイン不要で簡単）
- **C. LocalTunnel**（アカウント不要・完全無料）

In [ ]:
# ==============================
# A. ngrok で公開（無料版は URL が変わる）
# ==============================
from pyngrok import ngrok

NGROK_AUTH_TOKEN = ""  # 無料版でも必要なら設定

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

public_url = ngrok.connect(6080, "http")
print("ngrok public URL:", public_url)
print("ブラウザで開くとデスクトップが表示されます")


In [ ]:
# ==============================
# B. Cloudflare Tunnel で公開
# ==============================
!nohup sh -c "sleep 5 && cloudflared tunnel --url http://localhost:6080" > cloudflared.log 2>&1 &
import time
time.sleep(10)
!grep -oE 'https://[a-zA-Z0-9-]+\.trycloudflare\.com' cloudflared.log | head -1


In [ ]:
# ==============================
# C. LocalTunnel で公開
# ==============================
!npm install -g localtunnel
!nohup lt --port 6080 > localtunnel.log 2>&1 &
import time
time.sleep(5)
!cat localtunnel.log


## ④ PyAutoGUI でデスクトップを Python から操作

仮想デスクトップ上で以下を自動実行します。
1. `xterm` を開く（または `chromium-browser` を起動）
2. スクリーンショット取得
3. マウス移動・クリック

In [ ]:
import pyautogui
import time
import subprocess
from PIL import Image
import os

# 安全モード OFF（画面外でもエラーにしない）
pyautogui.FAILSAFE = False

# 仮想デスクトップ上で Chromium を起動（背景で）
subprocess.Popen([
    "chromium-browser",
    "--no-sandbox",
    "--disable-dev-shm-usage",
    "--disable-gpu",
    "--window-size=1280,720",
    "--window-position=0,0"
])
time.sleep(5)

# スクリーンショット 1: 起動直後のデスクトップ
screenshot1 = pyautogui.screenshot()
screenshot1.save("/home/user/outputs/desktop_start.png")
print("保存: desktop_start.png")

# マウスを中央に移動してクリック（Chromium ウィンドウをアクティブにする想定）
pyautogui.moveTo(640, 360, duration=1)
pyautogui.click()
time.sleep(1)

# 簡易的なキーボード操作: Ctrl+L (アドレスバー) -> URL 入力 -> Enter
pyautogui.keyDown('ctrl')
pyautogui.keyDown('l')
pyautogui.keyUp('l')
pyautogui.keyUp('ctrl')
time.sleep(0.5)

pyautogui.typewrite("https://www.google.com/search?q=pyautogui+colab", interval=0.01)
time.sleep(0.5)
pyautogui.press('return')
time.sleep(4)

# スクリーンショット 2: 検索結果画面
screenshot2 = pyautogui.screenshot()
screenshot2.save("/home/user/outputs/desktop_searched.png")
print("保存: desktop_searched.png")

# ファイルを表示（Colab なら自動表示される）
from IPython.display import display
display(screenshot2.resize((640, 360)))


## ⑤ Selenium でヘッドレス Chrome 操作

仮想デスクトップとは別に、Selenium 専用のヘッドレス Chrome を起動して、
ログインページのスクリーンショット取得などを行います。

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1280,720")

driver = webdriver.Chrome(options=options)

# 1. Google 検索
driver.get("https://www.google.com")
search_box = driver.find_element(By.NAME, "q")
search_box.send_keys("Selenium Colab GPU")
search_box.submit()

WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.ID, "search"))
)
driver.save_screenshot("/home/user/outputs/selenium_google.png")
print("保存: selenium_google.png")

# 2. GitHub ログインページのスクリーンショット（自動入力デモ）
driver.get("https://github.com/login")
time.sleep(2)
driver.find_element(By.ID, "login_field").send_keys("your_username")
driver.find_element(By.ID, "password").send_keys("your_password")
time.sleep(1)
driver.save_screenshot("/home/user/outputs/selenium_github_login.png")
print("保存: selenium_github_login.png")

driver.quit()
print("Selenium 処理完了")


## ⑥ Playwright で高速ブラウザ操作

Selenium より軽量で高速です。PDF 出力やモバイルエミュレーションも簡単です。

In [ ]:
from playwright.sync_api import sync_playwright

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True, args=["--no-sandbox"])
    context = browser.new_context(viewport={"width": 1280, "height": 720})
    page = context.new_page()

    page.goto("https://github.com/trending")
    page.wait_for_selector("article.Box-row")
    page.screenshot(path="/home/user/outputs/playwright_trending.png", full_page=True)
    print("保存: playwright_trending.png")

    # PDF 出力も可能（headless 必須）
    page.pdf(path="/home/user/outputs/github_trending.pdf", format="A4")
    print("保存: github_trending.pdf")

    browser.close()
print("Playwright 処理完了")


## ⑦ 画面録画（オプション）

仮想デスクトップ `:99` を ffmpeg で録画し、MP4 を出力します。
長時間録画するとファイルサイズが大きくなるので注意してください。

In [ ]:
import subprocess
import time

# 録画開始（バックグラウンド）
ffmpeg_proc = subprocess.Popen([
    "ffmpeg",
    "-f", "x11grab",
    "-video_size", "1280x720",
    "-i", ":99",
    "-codec:v", "libx264",
    "-preset", "ultrafast",
    "-r", "30",
    "-y",  # 上書き
    "/home/user/outputs/desktop_recording.mp4"
])

print("録画開始（10秒間デモ）...")
time.sleep(10)

# 録画停止
ffmpeg_proc.terminate()
try:
    ffmpeg_proc.wait(timeout=5)
except subprocess.TimeoutExpired:
    ffmpeg_proc.kill()

print("保存: desktop_recording.mp4")


## ⑧ クリーンアップ（セッション終了時）

プロセスを停止し、リソースを解放します。

In [ ]:
!pkill -f Xvfb
!pkill -f x11vnc
!pkill -f websockify
!pkill -f fluxbox
!pkill -f chromium-browser
!pkill -f ffmpeg
print("クリーンアップ完了")


## 付録: トラブルシューティング

- **noVNC のパスが見つからない**: `dpkg -L novnc | grep vnc.html` で実際のパスを確認
- **Chromium が起動しない**: `--no-sandbox` を必ず付ける
- **PyAutoGUI で画面が真っ黒**: `chromium-browser` が実際に起動しているか `ps aux | grep chromium` で確認
- **ngrok URL が変わる**: Hobby プランで固定ドメインを設定
- **Colab 90分で切断**: 長時間運用時は接続を維持する仕組み（JavaScript クリック等）が必要